<a href="https://colab.research.google.com/github/ethanresek/luminal-classifiers/blob/main/CS6140_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, RocCurveDisplay
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Imports OK")


In [ ]:
""" Include the below in any model files:
CSV = r'PATH/NAME/HERE'
DF = pd.read_csv(CSV)
Y_OLD_NAME = 'pam50_+_claudin-low_subtype'
KEEP = list(DF.columns[31:520]) + [Y_OLD_NAME]

Then preprocess:
pre_split = preprocess(DF, KEEP)
train, val, test = split_data(pre_split)
"""

def preprocess(df, keep, y_old_name='pam50_+_claudin-low_subtype', y_new_name='subtype',
               allowed=['LumA', 'LumB'], transform={'LumA': 1, 'LumB': 0}):
    df = df[keep].copy()
    df = df[df[y_old_name].isin(allowed)]
    df[y_old_name] = df[y_old_name].map(transform)
    df= df.rename(columns={y_old_name: y_new_name})
    return df

def split_data(df, test_size=0.15, val_size=0.176):
    train_val, test = train_test_split(df, test_size=test_size, random_state=42)
    train, val = train_test_split(train_val, test_size=val_size, random_state=42)
    return train, val, test


In [ ]:
# Load processed data
# Load the arrays saved after preprocessing
X = np.load("X.npy") # gene expression matrix, shape (1140, 489)
y = np.load("y.npy") # labels: 1 = LumA, 0 = LumB, shape (1140,)
gene_cols = np.load("gene_cols.npy", allow_pickle=True) # gene names, shape (489,)
# allow_pickle=True needed for string arrays

print(f"Data loaded")
print(f"X shape: {X.shape}") # should be (1140, 489)
print(f"y shape: {y.shape}") # should be (1140,)
print(f"LumA={np.sum(y==0)}, LumB={np.sum(y==1)}") # should be 679 / 461

# check 1: X and y must have the same number of rows
assert X.shape[0] == len(y), \
"X and y row count mismatch. Check that preprocessing saved matching arrays"

# check 2: number of gene names must match number of feature columns
assert X.shape[1] == len(gene_cols), \
"gene_cols length doesn't match X columns. Check preprocessing output"

# check 3: labels must be strictly 0 and 1 (nothing else)
assert set(np.unique(y)) == {0, 1}, \
"y should only contain 0 (LumA) and 1 (LumB). Check label encoding"

print("Checks OK")

In [ ]:
"""
Selects the top-k most differentially expressed genes using a two-sample t-test
between LumA and LumB patients.

This function must only ever receive TRAINING data. Passing in test data would
leak information about the test set into feature selection, inflating accuracy
estimates.

Args:
X_train: (n_train, n_genes) array of gene expression values
y_train: (n_train,) binary labels for training patients
k: number of top genes to select

Returns:
top_idx: (k,) array of column indices for the selected genes
"""

def select_top_genes(X_train, y_train, k):
    # Edge case: if k is larger than the number of available genes, just return
    # all gene indices, no selection needed
    if k >= X_train.shape[1]:
        return np.arange(X_train.shape[1])

    # Compute the t-statistic for every gene comparing LumA vs LumB
    # X_train[y_train == 0]: expression values for all LumA training patients
    # X_train[y_train == 1]: expression values for all LumB training patients
    # equal_var=False uses Welch's t-test, which doesn't assume equal variance
    t_stats = np.abs(
        stats.ttest_ind(
            X_train[y_train == 0],
            X_train[y_train == 1],
            equal_var=False
        ).statistic # .statistic extracts just the t-values, not the p-values
    )

    # np.argsort returns indices that would sort the array in ascending order
    # [::-1] reverses it to descending (highest t-stat first)
    # [:k] takes the top k indices
    return np.argsort(t_stats)[::-1][:k]

print("Feature selection function defined")

In [ ]:
"""
Builds an SVM classifier with RBF kernel.

Args:
C: regularization parameter. Higher C = less regularization, tighter fit to
training data. Lower C = smoother boundary. Default 1.0 is a reasonable starting
point.
gamma: RBF kernel bandwidth. 'scale' sets gamma = 1 / (n_features * X.var()),
which adapts automatically to the number of selected genes. This is better than
a fixed value across different panel sizes.

Returns:
sklearn SVC object (not yet fitted)
"""
def build_svm(C=1.0, gamma="scale"):
    return SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced",
        probability=True, random_state=RANDOM_SEED)

print("SVM builder defined")
print(f"Default params: C=1.0, gamma='scale', kernel='rbf'")

SVM builder defined
Default params: C=1.0, gamma='scale', kernel='rbf'


In [ ]:
# 5-fold CV across all panel sizes
PANEL_SIZES = [5, 10, 20, 50, 489] # 489 = all genes (full panel)
N_FOLDS = 5

# StratifiedKFold ensures each fold has the same ~60/40 LumA/LumB ratio
# shuffle=True randomizes patient order before splitting
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# results[panel_size] will be a list of metric dicts, one per fold
results = {k: [] for k in PANEL_SIZES}

# Outer loop: iterate over the 5 folds
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\n Fold {fold_idx + 1}/{N_FOLDS}")

    # Split full dataset into train and test arrays for this fold
    X_train_full = X[train_idx] # shape: (~912, 489)
    X_test_full = X[test_idx] # shape: (~228, 489)
    y_train = y[train_idx]
    y_test = y[test_idx]

    # Inner loop: test every panel size within this same fold
    for k in PANEL_SIZES:

        # Step 1: select top-k genes using TRAINING data only
        top_idx = select_top_genes(X_train_full, y_train, k)

        # Step 2: slice to selected genes
        X_train = X_train_full[:, top_idx] # shape: (~912, k)
        X_test = X_test_full[:, top_idx] # shape: (~228, k)

        # Step 3: scale, fit on train, apply to both
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # Step 4: train SVM
        svm = build_svm()
        svm.fit(X_train, y_train)

        # Step 5: predict on held-out test fold
        prob = svm.predict_proba(X_test)[:, 1] # LumB probabilities
        pred = (prob >= 0.5).astype(int) # threshold at 0.5 for hard labels

        # Step 6: record metrics for this fold + panel combination
        metrics = {
            "accuracy": accuracy_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
            # zero_division=0, returns 0 instead of warning if a class has no
            # predictions
            "auc": roc_auc_score(y_test, prob),
        }
        results[k].append(metrics)

        print(f"Panel={k:>3}  acc={metrics['accuracy']:.3f}  "
              f"f1={metrics['f1']:.3f}  auc={metrics['auc']:.3f}")

print("\n Cross-validation complete")
